# Phase 2: IndicTrans2 LoRA Fine-Tuning & CTranslate2 INT8 Quantization
### Hindi (`hin_Deva`) → Santhali Ol Chiki (`sat_Olck`) for Low-Resource Edge Android

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AshrafGalaxy/Vernacular_Pedagogy/blob/main/notebooks/colab_phase2_indictrans2_lora.ipynb)

**Objective:**
1. Pull authentic human bitext from **AI4Bharat BPCC** (`sat_Olck` <-> `hin_Deva`) and **IN22** benchmarks as specified in `Plan.md`.
2. Merge with verified Grade 1–3 NIPUN Bharat pedagogical domain seeds.
3. Fine-tune `ai4bharat/indictrans2-indic-indic-dist-320M` on Cloud GPU (Google Colab T4 / A100 free tier).
4. Merge LoRA adapter and quantize into **CTranslate2 INT8** format (~65 MB disk, ~120 MB RAM, ~40ms latency) for offline Android edge deployment.

## 1. Cloud GPU Verification & Environment Setup

> **IMPORTANT**: `transformers` is pinned to `>=4.39.0,<4.44.0` for IndicTransToolkit compatibility.
> Later versions (especially v5.x) break `tokenization_utils` imports used by IndicTransToolkit.

In [ ]:
# Verify GPU allocation
!nvidia-smi

# Install pinned dependencies (transformers version is CRITICAL for IndicTransToolkit)
!pip install -q 'transformers>=4.39.0,<4.44.0' datasets evaluate sacrebleu \
    peft bitsandbytes accelerate sentencepiece ctranslate2 huggingface_hub \
    sacremoses indic-nlp-library pandas

# Clone and install IndicTransToolkit
import os
if not os.path.exists('/content/IndicTransToolkit'):
    !git clone https://github.com/VarunGumma/IndicTransToolkit.git /content/IndicTransToolkit

# Idempotent patch: fix tokenization_utils import for transformers 4.39-4.43
COLLATOR = '/content/IndicTransToolkit/IndicTransToolkit/collator.py'
if os.path.exists(COLLATOR):
    with open(COLLATOR, 'r') as f:
        content = f.read()
    old_imp = 'from transformers.tokenization_utils import'
    new_imp = 'from transformers.tokenization_utils_base import'
    if old_imp in content and new_imp not in content:
        content = content.replace(old_imp, new_imp)
        with open(COLLATOR, 'w') as f:
            f.write(content)
        print('[PATCH] Fixed collator.py: tokenization_utils -> tokenization_utils_base')
    else:
        print('[PATCH] collator.py already correct. No changes needed.')

!pip install -q -e /content/IndicTransToolkit
print('\n✅ Environment setup complete!')

## 1.1. Hugging Face Authentication (Required for Gated Model)

The `ai4bharat/indictrans2-indic-indic-dist-320M` model is **gated**.
You must accept the terms at the [model page](https://huggingface.co/ai4bharat/indictrans2-indic-indic-dist-320M) and provide your token.

In [ ]:
import os

# Option 1: Set your token directly
HF_TOKEN = ""  # e.g. "hf_xxxxxxxxxxx"

# Option 2: Read from environment or persisted file
if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN and os.path.exists('/content/.hf_token'):
    with open('/content/.hf_token', 'r') as f:
        HF_TOKEN = f.read().strip()

# Authenticate
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('✅ Authenticated with Hugging Face Hub')
else:
    print('⚠️ No HF_TOKEN set. You may encounter 401 errors with gated models.')
    print('Set HF_TOKEN above or run: from huggingface_hub import login; login()')

AUTH_TOKEN = HF_TOKEN if HF_TOKEN else None

## 2. Clone Project Repository

In [ ]:
import os

REPO_URL = "https://github.com/AshrafGalaxy/Vernacular_Pedagogy.git"
PROJECT_DIR = "/content/Vernacular_Pedagogy"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull origin main

%cd {PROJECT_DIR}

## 2.1. Ingest Authentic AI4Bharat BPCC & IN22 Datasets (from Plan.md)
Downloads authentic human-translated parallel pairs from the official AI4Bharat repositories and merges them with our verified FLN database.

In [ ]:
# Ingest BPCC & IN22 datasets (best-effort: may fail if gated terms not accepted)
hf_arg = f'--hf-token {HF_TOKEN}' if HF_TOKEN else ''
!python scripts/01_fetch_bpcc_bitext.py {hf_arg}

# Generate train/val splits from FLN seeds + BPCC data
!python scripts/03_bitext_normalizer.py

train_tsv = "data/processed/bitext/train.tsv"
val_tsv = "data/processed/bitext/val.tsv"

assert os.path.exists(train_tsv), f"Missing {train_tsv}! Check normalizer output above."
assert os.path.exists(val_tsv), f"Missing {val_tsv}! Check normalizer output above."

import pandas as pd
train_df = pd.read_csv(train_tsv, sep='\t')
val_df = pd.read_csv(val_tsv, sep='\t')
print(f'\n✅ Training dataset: {len(train_df)} pairs')
print(f'✅ Validation dataset: {len(val_df)} pairs')
train_df.head()

## 3. Load IndicTrans2 Base Model & Tokenizer
Using the official distilled 320M Indic-to-Indic model: `ai4bharat/indictrans2-indic-indic-dist-320M`

In [ ]:
import sys
import types
import torch
import transformers
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Compatibility shim: transformers.onnx (removed in v5)
try:
    import transformers.onnx
except (ImportError, ModuleNotFoundError):
    onnx_mod = types.ModuleType('transformers.onnx')
    onnx_mod.OnnxConfig = object
    onnx_mod.OnnxSeq2SeqConfigWithPast = object
    sys.modules['transformers.onnx'] = onnx_mod
    print('[SHIM] Injected transformers.onnx stub')

# Compatibility shim: tokenization_utils.PreTrainedTokenizerBase
try:
    import transformers.tokenization_utils
    from transformers.tokenization_utils_base import PreTrainedTokenizerBase
    if not hasattr(transformers.tokenization_utils, 'PreTrainedTokenizerBase'):
        transformers.tokenization_utils.PreTrainedTokenizerBase = PreTrainedTokenizerBase
        print('[SHIM] Injected PreTrainedTokenizerBase')
except Exception as e:
    print(f'[SHIM WARN] {e}')

# Load IndicProcessor
if '/content/IndicTransToolkit' not in sys.path:
    sys.path.insert(0, '/content/IndicTransToolkit')
try:
    from IndicTransToolkit import IndicProcessor
except ImportError:
    from IndicTransToolkit.IndicTransToolkit import IndicProcessor

MODEL_NAME = "ai4bharat/indictrans2-indic-indic-dist-320M"
SRC_LANG = "hin_Deva"
TGT_LANG = "sat_Olck"

print("Loading tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, token=AUTH_TOKEN)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto",
    token=AUTH_TOKEN
)
ip = IndicProcessor(inference=False)
print(f"✅ Base model loaded on: {model.device}")

## 4. Parameter-Efficient Fine-Tuning (LoRA Configuration)
We train low-rank adaptation matrices on attention projections (`q_proj`, `k_proj`, `v_proj`, `out_proj`), freezing base weights to preserve broad multilingual generalization while tuning pedagogical accuracy.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj"],
    bias="none"
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

## 5. Dataset Pipeline & Tokenization

In [ ]:
import pandas as pd
from datasets import Dataset

def load_and_preprocess(tsv_path):
    df = pd.read_csv(tsv_path, sep="\t")
    # Drop rows with NaN values in source/target columns
    df = df.dropna(subset=['source', 'target'])
    src_sentences = df["source"].astype(str).tolist()
    tgt_sentences = df["target"].astype(str).tolist()
    assert len(src_sentences) > 0, f"No valid rows in {tsv_path}"
    
    # Preprocess via IndicProcessor (tagging with target language token)
    batch = ip.preprocess_batch(src_sentences, src_lang=SRC_LANG, tgt_lang=TGT_LANG)
    return Dataset.from_dict({"source": batch, "target": tgt_sentences})

train_ds = load_and_preprocess(train_tsv)
val_ds = load_and_preprocess(val_tsv)

def tokenize_fn(examples):
    model_inputs = tokenizer(examples["source"], max_length=128, truncation=True, padding="max_length")
    labels = tokenizer(text_target=examples["target"], max_length=128, truncation=True, padding="max_length")
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_ds.map(tokenize_fn, batched=True, remove_columns=["source", "target"])
tokenized_val = val_ds.map(tokenize_fn, batched=True, remove_columns=["source", "target"])
print(f"✅ Tokenized samples: train={len(tokenized_train)}, val={len(tokenized_val)}")

## 6. Model Fine-Tuning Execution

In [ ]:
import time
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

# Handle API differences between transformers versions
try:
    training_args = Seq2SeqTrainingArguments(
        output_dir="/content/indictrans2_sat_lora",
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=2,
        learning_rate=3e-4,
        num_train_epochs=3,
        fp16=torch.cuda.is_available(),
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        logging_steps=20,
        report_to="none"
    )
except TypeError:
    training_args = Seq2SeqTrainingArguments(
        output_dir="/content/indictrans2_sat_lora",
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=2,
        learning_rate=3e-4,
        num_train_epochs=3,
        fp16=torch.cuda.is_available(),
        evaluation_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        logging_steps=20,
        report_to="none"
    )

trainer = Seq2SeqTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer
)

print("🚀 Starting LoRA training...")
t0 = time.time()
trainer.train()
elapsed = (time.time() - t0) / 60
trainer.save_model("/content/indictrans2_sat_lora_final")
print(f"\n✅ LoRA training completed in {elapsed:.1f} minutes! Adapter saved.")

## 7. Merge LoRA Weights & Export Full PyTorch Model

In [ ]:
from peft import PeftModel

print("Merging LoRA weights with base model...")
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, trust_remote_code=True, token=AUTH_TOKEN)
merged_model = PeftModel.from_pretrained(base_model, "/content/indictrans2_sat_lora_final")
merged_model = merged_model.merge_and_unload()

EXPORT_DIR = "/content/indictrans2_sat_merged"
merged_model.save_pretrained(EXPORT_DIR)
tokenizer.save_pretrained(EXPORT_DIR)
print(f"✅ Merged model exported to: {EXPORT_DIR}")

## 8. CTranslate2 INT8 Quantization (Edge Engine Conversion)
Converts the PyTorch model into a lightweight, highly optimized INT8 CTranslate2 engine (~65 MB) runnable offline on low-end Android CPUs.

In [ ]:
!python -m ctranslate2.converters.transformers \
    --model /content/indictrans2_sat_merged \
    --output_dir /content/indictrans2_sat_int8_ct2 \
    --quantization int8 \
    --trust_remote_code \
    --low_cpu_mem_usage

!ls -lh /content/indictrans2_sat_int8_ct2
print('\n✅ CTranslate2 INT8 model generated!')

## 9. Benchmark Edge CTranslate2 INT8 Inference Latency

In [ ]:
import time
import ctranslate2

# Reload IndicProcessor in inference mode for benchmarking
ip_infer = IndicProcessor(inference=True)

translator = ctranslate2.Translator("/content/indictrans2_sat_int8_ct2", device="cpu", compute_type="int8")
test_sentences = [
    "साफ़ पानी पियो",
    "किताब खोलो",
    "यहाँ पाँच सेब हैं",
    "चलो हम सब मिलकर पढ़ें",
    "बस्ते से पेंसिल निकालो",
    "गाय घास खा रही है"
]

print("Benchmarking CPU INT8 Inference Latency:")
print("-" * 70)
for s in test_sentences:
    preprocessed = ip_infer.preprocess_batch([s], src_lang=SRC_LANG, tgt_lang=TGT_LANG)
    tokens = tokenizer.convert_ids_to_tokens(tokenizer.encode(preprocessed[0]))
    
    t0 = time.perf_counter()
    results = translator.translate_batch([tokens])
    t_elapsed = (time.perf_counter() - t0) * 1000
    
    out_tokens = results[0].hypotheses[0]
    out_ids = tokenizer.convert_tokens_to_ids(out_tokens)
    out_text = tokenizer.decode(out_ids, skip_special_tokens=True)
    # Post-process with IndicProcessor
    out_text = ip_infer.postprocess_batch([out_text], lang=TGT_LANG)[0]
    print(f"  '{s}' -> '{out_text}' (Latency: {t_elapsed:.1f} ms)")

print("-" * 70)
print("✅ Benchmark complete!")

## 10. Archive & Download Edge Model

In [ ]:
%%bash
cd /content
tar -czf indictrans2_sat_int8_ct2.tar.gz indictrans2_sat_int8_ct2/
ls -lh indictrans2_sat_int8_ct2.tar.gz
echo "✅ INT8 Edge MT Model ready for Android asset packaging!"

In [ ]:
# Download to local machine (Colab only)
try:
    from google.colab import files
    files.download('/content/indictrans2_sat_int8_ct2.tar.gz')
except ImportError:
    print('Not running in Colab. Archive is at /content/indictrans2_sat_int8_ct2.tar.gz')